How many Review-Commons-reviewed preprints have reviews/author replies without DOIs?

In [1]:
import requests
import pandas

In [2]:
dois_by_reviewing_service = requests.get("https://eeb.embo.org/api/v1/by_reviewing_service/").json()
revcom_data = [
    s
    for s in dois_by_reviewing_service
    if s["id"] == "review commons"
][0]
revcom_data

{'id': 'review commons',
 'papers': [{'rank': '',
   'pub_date': '2025-04-25T00:00:00Z',
   'slug': 'Sex-lethal-is-recruited-to-chromatin-to-promote-neuronal-tRNA-synthesis-in-males-through-RNA-Polymerase-III-regulation',
   'doi': '10.1101/2025.04.25.650657'},
  {'rank': '',
   'pub_date': '2020-01-11T00:00:00Z',
   'slug': 'Integrative-analysis-of-large-scale-loss-of-function-screens-identifies-robust-cancer-associated-genetic-interactions',
   'doi': '10.1101/646810'},
  {'rank': '',
   'pub_date': '2020-05-07T00:00:00Z',
   'slug': 'Transcriptional-comparison-of-Testicular-Adrenal-Rest-Tumors-with-fetal-and-adult-tissues',
   'doi': '10.1101/2020.05.07.082313'},
  {'rank': '',
   'pub_date': '2020-04-23T00:00:00Z',
   'slug': 'ESI-mutagenesis-A-one-step-method-for-introducing-point-mutations-into-bacterial-artificial-chromosome-transgenes',
   'doi': '10.1101/844282'},
  {'rank': '',
   'pub_date': '2020-05-22T00:00:00Z',
   'slug': 'A-large-accessory-protein-interactome-is-rewired

In [3]:
pandas.DataFrame([(p["doi"], p["pub_date"]) for p in revcom_data["papers"]], columns=["doi", "published_at"])

,doi,published_at
0,10.1101/2025.04.25.650657,2025-04-25T00:00:00Z
1,10.1101/646810,2020-01-11T00:00:00Z
2,10.1101/2020.05.07.082313,2020-05-07T00:00:00Z
3,10.1101/844282,2020-04-23T00:00:00Z
4,10.1101/2020.05.20.106583,2020-05-22T00:00:00Z
...,...,...
1241,10.1101/2024.08.17.608283,2024-08-17T00:00:00Z
1242,10.1101/2024.01.06.574461,2024-01-08T00:00:00Z
1243,10.1101/2024.05.02.592277,2025-02-05T00:00:00Z
1244,10.1101/2024.08.19.608731,2024-08-20T00:00:00Z


In [4]:
request_payload = {"dois": [p["doi"] for p in revcom_data["papers"]]}
data = requests.post("https://eeb.embo.org/api/v1/dois/", json=request_payload).json()
len(data), data[:3]

(1246,
 [{'id': 13735762,
   'doi': '10.1101/2024.12.09.627460',
   'version': '1.2',
   'source': 'f755ded0-6c89-1014-8d6f-a90e8b422d8b.meca',
   'journal': 'bioRxiv',
   'title': 'Deciphering the CD73⁺ Regulatory γδ T Cell ecosystem associated with poor survival in Ovarian cancer',
   'abstract': 'The ability of tumor cells to overcome immune surveillance is an essential step in tumor development and progression. Among the immune cells playing a role in tumor control, γδ T cells contribute to the immune response against many tumor types through their direct cytotoxic activity against cancer cells and their capacity to regulate the functions of other immune cells. However, their presence in the tumor microenvironment is also associated with poor prognosis, suggesting that γδ T cells may also have pro-tumor activities. We previously described a regulatory γδ T-cell subset that expresses CD73 and produces IL-10, IL-8 and adenosine. Here, we report a higher CD73+ γδ T cell density in the

In [5]:
reviews = [(a["doi"], r) for a in data for r in a["review_process"]["reviews"]]
responses = [(a["doi"], a["review_process"]["response"]) for a in data if a["review_process"]["response"] is not None]
len(reviews), len(responses)

(3597, 1241)

In [6]:
df = pandas.DataFrame(
    [
        (doi, r.get("doi", None), r["posting_date"])
        for doi, r in reviews + responses
    ],
    columns=["article_doi", "doi", "posting_date"]
)
df

,article_doi,doi,posting_date
0,10.1101/2024.12.09.627460,10.15252/rc.2025199660,2025-07-16T12:12:43.467979+00:00
1,10.1101/2024.12.09.627460,10.15252/rc.2025017913,2025-07-16T12:12:44.099831+00:00
2,10.1101/2024.12.09.627460,10.15252/rc.2025902375,2025-07-16T12:12:44.677135+00:00
3,10.1101/2024.01.20.573595,10.15252/rc.2024583495,2024-10-29T08:13:07.493621+00:00
4,10.1101/2024.01.20.573595,10.15252/rc.2024264773,2024-10-29T08:13:07.911046+00:00
...,...,...,...
4833,10.1101/2019.12.17.877639,10.15252/rc.2022106179,2020-03-09T17:17:30.246842+00:00
4834,10.1101/856773,10.15252/rc.2022058024,2020-02-05T11:16:05.099329+00:00
4835,10.1101/801845,10.15252/rc.2022491265,2020-03-27T10:22:42.778810+00:00
4836,10.1101/636449,10.15252/rc.2022543615,2020-04-20T10:17:34.902431+00:00


In [7]:
df[df["doi"].isna()]

,article_doi,doi,posting_date
72,10.1101/2025.05.13.653105,None,2025-08-07T08:12:54.959599+00:00
73,10.1101/2025.05.13.653105,None,2025-08-07T08:12:55.455253+00:00
74,10.1101/2025.05.13.653105,None,2025-08-07T08:12:55.767204+00:00
81,10.1101/2025.05.09.653054,None,2025-08-07T13:12:46.596786+00:00
82,10.1101/2025.05.09.653054,None,2025-08-07T13:12:47.082470+00:00
...,...,...,...
4610,10.1101/2020.11.19.389544,None,2023-02-24T09:43:18.206218+00:00
4654,10.1101/2021.04.26.441408,None,2023-11-14T15:18:04.614783+00:00
4701,10.1101/2021.11.26.470092,None,2023-11-14T15:17:49.330376+00:00
4712,10.1101/2021.10.27.466115,None,2023-11-14T15:17:51.546792+00:00


In [8]:
df[df["doi"].isna()]["article_doi"].value_counts()

10.1101/2023.05.29.542713    7
10.1101/2023.07.18.549574    5
10.1101/2023.12.25.573293    5
10.1101/2024.10.29.620637    5
10.1101/2022.10.17.512562    5
                            ..
10.1101/2022.07.15.500272    3
10.1101/2022.07.11.499563    3
10.1101/2025.05.09.653054    3
10.1101/2020.09.29.318857    3
10.1101/2023.07.07.548114    1
Name: article_doi, Length: 93, dtype: int64